In [12]:
import torch 
from torch import nn 

In [13]:
X = torch.tensor([
    [1,2,3],
    [2,3,4],
    [4,5,6]
],
dtype=torch.float)  

y = torch.tensor([
    3,
    5, 
    7
],
dtype=torch.float)

In [19]:
class MyNeuralNetwork(nn.Module):
    
    def __init__(self):
        super().__init__() 

        self.net = nn.Sequential(
            nn.Linear(3, 8),   # input (3 features) -> 8 (Hidden Layers)
            nn.ReLU(),          # ReLu Layer After Hidden Layer 
            nn.Linear(8, 1)     # Hidden (8 Layer After Relu)  -> 1 output Linear  
        )
    
    def forward(self, X):
        return self.net(X)  

In [21]:
model = MyNeuralNetwork() 
loss_fn = nn.MSELoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)  

print("--Training--") 

for epoch in range(100):

    y_pred = model(X) 

    loss = loss_fn(y_pred, y)
    
    optimizer.zero_grad() 

    loss.backward() 
    optimizer.step()     

print(f"Final training Loss: {loss.item()}")
print("Predictions before saving:\n", model(X).detach())

--Training--
Final training Loss: 2.7463889122009277
Predictions before saving:
 tensor([[4.6561],
        [4.8653],
        [5.2837]])


In [22]:
# Save only the learned weights and biases
torch.save(model.state_dict(), "simple_model.pth")
print("\nModel weights successfully saved to simple_model.pth")


Model weights successfully saved to simple_model.pth


In [23]:
loaded_model = MyNeuralNetwork() 
state_dict = torch.load("simple_model.pth", weights_only=True)

loaded_model.load_state_dict(state_dict)
loaded_model.eval()

# 5. Run prediction with autograd disabled
with torch.no_grad():
    new_input = torch.tensor([[1.0, 2.0, 3.0]], dtype=torch.float32)
    prediction = loaded_model(new_input)
    print("\nInference Output on loaded model:", prediction.item())


Inference Output on loaded model: 4.656087875366211


In [24]:
# Save Checkpoint
checkpoint = {
    "epoch": 200,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": loss.item()
}
torch.save(checkpoint, "checkpoint.pth")

# Load Checkpoint
checkpoint = torch.load("checkpoint.pth", weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
start_epoch = checkpoint["epoch"]